In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("D:/Kalika per.projects/cs-training.csv")

# remove unwanted column
if "Unnamed: 0" in df.columns:
    df.drop("Unnamed: 0", axis=1, inplace=True)

df.fillna(df.median(), inplace=True)

In [4]:
df.head()

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [5]:
X = df.drop("SeriousDlqin2yrs", axis=1)
y = df["SeriousDlqin2yrs"]

print("Feature count:", X.shape[1])  # MUST BE 10

Feature count: 10


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
from imblearn.over_sampling import SMOTE

sm = SMOTE()
X_train, y_train = sm.fit_resample(X_train, y_train)

In [8]:
from xgboost import XGBClassifier

best_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

best_model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [9]:
best_model.predict_proba(X_test[:1])

array([[0.9557146 , 0.04428543]], dtype=float32)

In [9]:
from joblib import dump

dump(best_model, "model.pkl")

In [10]:
import shap
explainer = shap.Explainer(best_model)

In [16]:
while True:
    print("\n🔹 Credit Risk Prediction (Model-Based)\n")

    try:
        revolving = float(input("Revolving Utilization (0–1): "))
        age = int(input("Age: "))

        print("\n💡 Debt Ratio Calculator")
        total_debt = float(input("Total monthly debt: "))
        income = float(input("Monthly income: "))

        if income == 0:
            print("❌ Income cannot be zero")
            continue

        debt = total_debt / income
        print(f"👉 Debt Ratio: {debt:.2f}")

        late_90 = int(input("90 Days Late: "))

        # 🔥 IMPORTANT: EXACT 10 FEATURES (MATCH MODEL)
        input_data = np.array([[ 
            revolving,
            age,
            0,
            debt,
            income,
            5,
            late_90,
            1,
            0,
            1
        ]], dtype=float)

        print("\nModel expects:", best_model.n_features_in_)
        print("Input given:", input_data.shape[1])

        # =========================
        # PREDICTION (MODEL ONLY)
        # =========================
        probability = best_model.predict_proba(input_data)[0][1]
        risk_percent = probability * 100

        print("\n📊 Risk Score:")

        if risk_percent >= 75:
            print(f"🚨 VERY HIGH RISK ({risk_percent:.1f}%)")
        elif risk_percent >= 50:
            print(f"⚠️ HIGH RISK ({risk_percent:.1f}%)")
        elif risk_percent >= 30:
            print(f"⚠️ MODERATE RISK ({risk_percent:.1f}%)")
        else:
            print(f"✅ LOW RISK ({risk_percent:.1f}%)")

        # =========================
        # SHAP EXPLANATION
        # =========================
        shap_values = explainer(input_data)

        print("\n🧠 Why this prediction?")

        feature_names = X.columns
        impacts = shap_values.values[0]

        sorted_idx = np.argsort(np.abs(impacts))[::-1]

        for i in sorted_idx[:3]:
            if impacts[i] > 0:
                print(f"⬆️ {feature_names[i]} increases risk")
            else:
                print(f"⬇️ {feature_names[i]} reduces risk")

        # =========================
        # SUGGESTIONS
        # =========================
        print("\n💡 Recommendation:")

        if risk_percent >= 50:
            print("👉 Reduce debt and avoid delayed payments")
            print("👉 Improve repayment history")
        else:
            print("👉 Maintain financial discipline")
            print("👉 Keep debt low")

        print("\n✅ Analysis completed.")

    except Exception as e:
        print("\n❌ Error:", e)

    cont = input("Do another analysis? (yes/no): ").lower()

    if cont != "yes":
        print("\n👋 Done. Thank you!")
        break


🔹 Credit Risk Prediction (Model-Based)



Revolving Utilization (0–1):  0.7
Age:  20



💡 Debt Ratio Calculator


Total monthly debt:  60000
Monthly income:  100000


👉 Debt Ratio: 0.60


90 Days Late:  0



Model expects: 10
Input given: 10

📊 Risk Score:
⚠️ MODERATE RISK (37.1%)

🧠 Why this prediction?
⬆️ RevolvingUtilizationOfUnsecuredLines increases risk
⬇️ NumberOfDependents reduces risk
⬆️ DebtRatio increases risk

💡 Recommendation:
👉 Maintain financial discipline
👉 Keep debt low

✅ Analysis completed.


Do another analysis? (yes/no):  no



👋 Done. Thank you!


In [13]:
!pip install streamlit

  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/9.1 MB 6.2 MB/s eta 0:00:02
   ---------- ----------------------------- 2.4/9.1 MB 6.0 MB/s eta 0:00:02
   ---------------- ----------------------- 3.7/9.1 MB 5.7 MB/s eta 0:00:01
   -------------------- ------------------- 4.7/9.1 MB 5.5 MB/s eta 0:00:01
   ------------------------- -------------- 5.8/9.1 MB 5.4 MB/s eta 0:00:01
   ------------------------------ --------- 6.8/9.1 MB 5.4 MB/s eta 0:00:01
   ---------------------------------- ----- 7.9/9.1 MB 5.3 MB/s eta 0:00:01
   ------------------------------------- -- 8.4/9.1 MB 5.0 MB/s eta 0:0

In [1]:
from joblib import dump
dump(best_model, "model.pkl")

NameError: name 'best_model' is not defined